# 컴퓨터비전 과제 #2

## 과제#1 이후부터 현재까지 실습 코드를 모두 수행하고 결과 출력 (2점)

- 각 수행마다 코드에는 어떤 과정인지 주석 처리

**구성**
1. 해리스 코너 검출 (Harris Corner Detection)
2. 슈퍼픽셀 분할 (SLIC Superpixel)
3. 최적화 분할 (SLIC + N-Cut)
4. 영상 품질 측정 (PSNR & SSIM)
5. SIFT vs ORB Feature Matching Benchmark (BF/FLANN)
6. ORB 기반 비디오 객체 추적
7. RANSAC 기반 호모그래피 추정 (SIFT + Faiss)
8. Stereo Block Matching (StereoBM, SAD 기반)
9. 2.5D 영상 (Disparity → Point Cloud → PLY 저장)

## 공통 라이브러리 import

이후 모든 실습에서 공통적으로 사용하는 패키지를 한 번에 불러옵니다.

In [ ]:
import cv2
import numpy as np
import time
import urllib.request
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

## [실습 1] 해리스 코너 검출 (Harris Corner Detection)

**목표:** 영상의 변화량이 모든 방향으로 큰 지점을 코너로 식별하는 해리스 코너 검출 알고리즘을 적용하고, 임계값 기반으로 코너점을 시각화합니다.

In [ ]:
# 1. 샘플 이미지 다운로드 (스도쿠 이미지: 직선/코너 풍부)
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/sudoku.png'
urllib.request.urlretrieve(url, 'sudoku.png')
img = cv2.imread('sudoku.png')

# 2. 코너 검출 전처리 (그레이스케일 + float32 변환)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = np.float32(gray)

# 3. 해리스 코너 검출 수행
# cv2.cornerHarris(입력영상, 블록크기, 소벨커널크기, k값)
# - 2: 블록 크기 (코너 검출용 윈도우)
# - 3: 소벨 커널 크기 (미분 연산자 크기)
# - 0.04: Harris 코너 응답 함수의 k값 (보통 0.04~0.06)
dst = cv2.cornerHarris(gray, 2, 3, 0.04)

# 4. 검출된 코너 점들을 시각적으로 강조 (팽창)
dst = cv2.dilate(dst, None)

# 5. 임계값(가장 강한 응답값의 1%) 이상인 픽셀만 코너로 채택 후 빨간색 표시
img_result = img.copy()
threshold = 0.01 * dst.max()
img_result[dst > threshold] = [0, 0, 255]

# 6. 결과 출력
print('--- [실습 1 결과: 원본 이미지] ---')
cv2_imshow(img)
print('\n--- [실습 1 결과: 해리스 코너 검출 결과 (빨간색 점)] ---')
cv2_imshow(img_result)

## [실습 2] 슈퍼픽셀 분할 (SLIC)

**목표:** SLIC(Simple Linear Iterative Clustering) 알고리즘을 사용해 영상을 의미 있는 작은 단위(Superpixel)로 분할하고, 각 영역을 평균 색상으로 채워 시각화합니다.

In [ ]:
from skimage.segmentation import slic, mark_boundaries
from skimage.color import label2rgb

# 1. 샘플 이미지 다운로드 및 RGB 변환
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/butterfly.jpg'
urllib.request.urlretrieve(url, 'butterfly.jpg')
img = cv2.imread('butterfly.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. SLIC 알고리즘 적용
# n_segments: 분할할 슈퍼픽셀 개수
# compactness: 색상 유사도 vs 공간 거리의 가중치 (높을수록 정사각형에 가까움)
segments = slic(img_rgb, n_segments=400, compactness=10, sigma=1, start_label=1)

# 3. 각 슈퍼픽셀 영역을 해당 영역의 평균 색상으로 채움
superpixel_avg = label2rgb(segments, img_rgb, kind='avg')

# 4. 결과 시각화 (원본 / 경계선 / 평균 색상)
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.title('1. Original (Butterfly)')
plt.imshow(img_rgb)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title('2. SLIC Boundaries (400 segments)')
plt.imshow(mark_boundaries(img_rgb, segments))
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title('3. Segmented (Avg Color)')
plt.imshow(superpixel_avg.astype('uint8'))
plt.axis('off')

plt.tight_layout()
print('--- [실습 2 결과: SLIC Superpixel] ---')
plt.show()

## [실습 3] 최적화 분할 (SLIC + N-Cut)

**목표:** SLIC으로 영상을 잘게 쪼갠 후 N-Cut(Normalized Cut) 알고리즘으로 유사한 영역을 병합하여 최적화된 분할 결과를 얻습니다.

In [ ]:
from skimage import graph  # N-Cut 계산용 그래프 모듈

# 1. 샘플 이미지 로드 (실습 2와 동일한 butterfly)
img = cv2.imread('butterfly.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. SLIC 슈퍼픽셀 분할 (Oversegmentation, 400조각)
labels = slic(img_rgb, n_segments=400, compactness=10, sigma=1, start_label=1)

# 3. 슈퍼픽셀 간 인접 관계와 평균 색상 차이로 RAG(Region Adjacency Graph) 생성
g = graph.rag_mean_color(img_rgb, labels)

# 4. N-Cut(Normalized Cut) 적용
# thresh: 잘라낼 기준값 (낮을수록 더 잘게, 높을수록 크게 뭉침)
# num_cuts: 반복 절단 횟수
nc_labels = graph.cut_normalized(labels, g, thresh=1.0, num_cuts=2)

# 5. 시각화 데이터 준비 (각 영역 평균 색상)
slic_avg = label2rgb(labels, img_rgb, kind='avg')
ncut_avg = label2rgb(nc_labels, img_rgb, kind='avg')

# 6. 결과 출력 (원본 / SLIC 경계 / SLIC 평균 / N-Cut 결과)
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.title('1. Original')
plt.imshow(img_rgb)
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title('2. SLIC Boundaries')
plt.imshow(mark_boundaries(img_rgb, labels))
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title('3. SLIC Avg Color')
plt.imshow(slic_avg.astype('uint8'))
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title('4. N-Cut Result')
plt.imshow(ncut_avg.astype('uint8'))
plt.axis('off')

plt.tight_layout()
print('--- [실습 3 결과: SLIC + N-Cut 최적화 분할] ---')
plt.show()

## [실습 4] 영상 품질 측정 (PSNR & SSIM)

**목표:** 원본을 1/4로 축소 후 다시 확대했을 때 발생하는 화질 저하를 PSNR(Peak Signal-to-Noise Ratio)과 SSIM(Structural Similarity Index Measure)으로 정량 측정합니다.

In [ ]:
from skimage.metrics import structural_similarity as ssim

# 1. 샘플 이미지 다운로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/baboon.jpg'
urllib.request.urlretrieve(url, 'baboon.jpg')
original = cv2.imread('baboon.jpg')

# 2. 영상 축소 후 재확대 (보간법 적용)
height, width = original.shape[:2]

# 1/4 크기로 축소 (INTER_AREA: 다운샘플링에 적합)
small = cv2.resize(original, (width//4, height//4), interpolation=cv2.INTER_AREA)

# 다시 원래 크기로 확대 (INTER_LINEAR)
restored = cv2.resize(small, (width, height), interpolation=cv2.INTER_LINEAR)

# 3. 품질 측정
# PSNR: 픽셀 오차 기반 (높을수록 원본과 유사)
psnr_val = cv2.PSNR(original, restored)
# SSIM: 구조적 유사성 기반 (1에 가까울수록 유사)
ssim_val = ssim(original, restored, channel_axis=2)

# 4. 결과 출력
print('--- [실습 4 결과: 화질 지표] ---')
print(f'PSNR: {psnr_val:.2f} dB')
print(f'SSIM: {ssim_val:.4f}')
print('-' * 30)

# 5. 원본과 복원본을 가로로 붙여 시각 비교
combined = np.hstack((original, restored))
print('\n[왼쪽: 원본 | 오른쪽: 1/4 축소 후 복원본]')
cv2_imshow(cv2.resize(combined, (0, 0), fx=0.8, fy=0.8))

## [실습 5] Feature Matching Benchmark (SIFT vs ORB, BF vs FLANN)

**목표:** SIFT와 ORB 특징점 검출기를 BF(Brute-Force) 매처와 FLANN 매처에 각각 조합하여 검출 시간/매칭 시간/매칭 개수를 벤치마크합니다.

In [ ]:
# 1. Aloe 스테레오 페어 다운로드 및 흑백 변환 (서로 다른 시점에서 촬영된 이미지 두 장)
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeL.jpg'
urllib.request.urlretrieve(url, 'aloeL.jpg')
img1 = cv2.imread('aloeL.jpg', cv2.IMREAD_GRAYSCALE)

url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeR.jpg'
urllib.request.urlretrieve(url, 'aloeR.jpg')
img2 = cv2.imread('aloeR.jpg', cv2.IMREAD_GRAYSCALE)

In [ ]:
def benchmark_full(name, detector, matcher_type='BF', ratio_threshold=0.7):
    # 1. 특징점 검출 + 기술자 생성 시간 측정
    start_time = time.time()
    kp1, des1 = detector.detectAndCompute(img1, None)
    kp2, des2 = detector.detectAndCompute(img2, None)
    detect_time = (time.time() - start_time) * 1000

    # 2. 매처(Matcher) 설정
    # SIFT는 L2 norm, ORB는 Hamming distance 사용
    if matcher_type == 'BF':
        norm = cv2.NORM_L2 if name == 'SIFT' else cv2.NORM_HAMMING
        matcher = cv2.BFMatcher(norm, crossCheck=False)
    else:
        # FLANN 매처 파라미터: SIFT는 KDTree, ORB는 LSH 사용
        if name == 'SIFT':
            index_params = dict(algorithm=1, trees=5)  # FLANN_INDEX_KDTREE
        else:
            index_params = dict(algorithm=6, table_number=6,
                                key_size=12, multi_probe_level=1)
        search_params = dict(checks=50)
        matcher = cv2.FlannBasedMatcher(index_params, search_params)

    # 3. k-NN 매칭 (k=2) 후 Lowe's Ratio Test로 좋은 매칭만 추출
    start_time = time.time()
    matches = matcher.knnMatch(des1, des2, k=2)
    good_matches = []
    for m_n in matches:
        if len(m_n) == 2:
            m, n = m_n
            if m.distance < ratio_threshold * n.distance:
                good_matches.append(m)
    match_time = (time.time() - start_time) * 1000

    print(f'\n--- [{name} + {matcher_type}] ---')
    print(f'Detect: {detect_time:.1f}ms | Match: {match_time:.1f}ms | Goods: {len(good_matches)}')

    # 4. 매칭 결과 시각화 (상위 50개)
    res_img = cv2.drawMatches(img1, kp1, img2, kp2,
                              good_matches[:50], None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    info_text = f'{name}+{matcher_type}: {len(good_matches)} matches'
    cv2.putText(res_img, info_text, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2_imshow(res_img)

    return kp1, kp2, good_matches

In [ ]:
# SIFT 벤치마크 (BF + FLANN)
sift = cv2.SIFT_create()

print('--- [실습 5 결과: SIFT 벤치마크 시작] ---')
benchmark_full('SIFT', sift, 'BF')
benchmark_full('SIFT', sift, 'FLANN')

In [ ]:
# ORB 벤치마크 (BF + FLANN). ORB는 이진 기술자라 ratio threshold를 0.85로 완화
orb = cv2.ORB_create(nfeatures=2000)

print('--- [실습 5 결과: ORB 벤치마크 시작] ---')
benchmark_full('ORB', orb, 'BF', ratio_threshold=0.85)
benchmark_full('ORB', orb, 'FLANN', ratio_threshold=0.85)

## [실습 6] ORB 기반 비디오 객체 추적

**목표:** ORB 특징점과 BF 매처를 활용하여 비디오의 첫 프레임에서 설정한 타겟 영역을 매 프레임에서 추적하고, 검출/매칭 속도 및 FPS를 측정합니다.

In [ ]:
# 1. 샘플 영상 로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi'
urllib.request.urlretrieve(url, 'vtest.avi')
cap = cv2.VideoCapture('vtest.avi')

# 2. 첫 프레임에서 추적할 타겟 영역(ROI) 설정
ret, first_frame = cap.read()
if not ret:
    raise RuntimeError('첫 프레임을 읽을 수 없습니다.')

target_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
target_roi = target_gray[100:300, 200:400]  # 보행자가 있는 영역

# 3. ORB 검출기 및 BF 매처 초기화 (Hamming distance, ORB는 이진 기술자이므로)
orb = cv2.ORB_create(nfeatures=1000)
kp1, des1 = orb.detectAndCompute(target_roi, None)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

In [ ]:
# 4. 프레임별 추적 및 속도(ms, FPS) 측정
print('--- [실습 6 결과: ORB Video Tracking 속도 측정] ---')
print(f"{'Frame':<8} | {'Detect(ms)':<12} | {'Match(ms)':<12} | {'Total(ms)':<12} | {'FPS':<6}")
print('-' * 65)

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # --- 속도 측정 시작 ---
    start_total = time.time()

    # 4-1. 특징점 검출 + 기술자 생성 시간 측정
    start_det = time.time()
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    kp2, des2 = orb.detectAndCompute(gray_frame, None)
    end_det = time.time()
    det_time = (end_det - start_det) * 1000

    match_time = 0
    good_matches = []

    if des2 is not None:
        # 4-2. 매칭 시간 측정 (KNN + Ratio Test)
        start_match = time.time()
        matches = bf.knnMatch(des1, des2, k=2)
        good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
        end_match = time.time()
        match_time = (end_match - start_match) * 1000

    # --- 속도 측정 종료 ---
    end_total = time.time()
    total_time = (end_total - start_total) * 1000
    fps = 1.0 / (end_total - start_total) if total_time > 0 else 0

    # 4-3. 매 10프레임마다 지표 출력
    if frame_count % 10 == 0:
        print(f'{frame_count:<8} | {det_time:<12.2f} | {match_time:<12.2f} | {total_time:<12.2f} | {fps:<6.1f}')

    # 4-4. 매 100프레임마다 매칭 결과 시각화
    if frame_count % 100 == 0:
        res = cv2.drawMatches(target_roi, kp1, frame, kp2,
                              good_matches[:20], None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        cv2.putText(res, f'FPS: {fps:.1f}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2_imshow(res)

    frame_count += 1

cap.release()
print(f'\nTotal frames processed: {frame_count}')

## [실습 7] RANSAC 기반 호모그래피 추정

**목표:** SIFT로 추출한 특징점을 Faiss-GPU 기반 KNN 매칭(또는 BF 매칭) + Lowe's Ratio Test로 1차 필터링한 뒤, RANSAC으로 호모그래피 행렬을 추정해 outlier를 제거합니다.

*참고: Faiss-GPU 미설치 환경에서도 동작하도록 BF 매처 fallback을 추가했습니다.*

In [ ]:
# 1. 이미지 로드 (서로 다른 시점에서 촬영된 두 장)
url1 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeL.jpg'
url2 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeR.jpg'
urllib.request.urlretrieve(url1, 'aloeL.jpg')
urllib.request.urlretrieve(url2, 'aloeR.jpg')

img1_gray = cv2.imread('aloeL.jpg', cv2.IMREAD_GRAYSCALE)
img2_gray = cv2.imread('aloeR.jpg', cv2.IMREAD_GRAYSCALE)

# 2. SIFT 특징점 추출 (반복 패턴 영상이므로 nfeatures를 충분히 크게 설정)
sift = cv2.SIFT_create(nfeatures=5000)
kp1, des1 = sift.detectAndCompute(img1_gray, None)
kp2, des2 = sift.detectAndCompute(img2_gray, None)

print(f'특징점 개수: Image1({len(kp1)}개), Image2({len(kp2)}개)\n')

# 3. Faiss-GPU 기반 고속 KNN(K=2) 매칭. 미설치/미지원시 BF 매처로 fallback
good_matches = []
try:
    import faiss
    res = faiss.StandardGpuResources()

    des1_f32 = des1.astype('float32')
    des2_f32 = des2.astype('float32')

    index = faiss.IndexFlatL2(128)
    gpu_index = faiss.index_cpu_to_gpu(res, 0, index)
    gpu_index.add(des2_f32)
    distances, indices = gpu_index.search(des1_f32, 2)

    # 4. Lowe's Ratio Test로 1차 필터링
    for i in range(len(des1)):
        if distances[i][0] < 0.5 * distances[i][1]:
            m = cv2.DMatch(_queryIdx=i,
                           _trainIdx=int(indices[i][0]),
                           _distance=float(distances[i][0]))
            good_matches.append(m)
    print(f'Faiss 매칭 완료 | Ratio Test 통과: {len(good_matches)}개')
except Exception as e:
    # Faiss 미사용 환경에서는 BF 매처로 동일 절차 수행
    print(f'[알림] Faiss 사용 불가 ({e.__class__.__name__}). BF 매처로 대체합니다.')
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    knn = bf.knnMatch(des1, des2, k=2)
    for m, n in knn:
        if m.distance < 0.5 * n.distance:
            good_matches.append(m)
    print(f'BF 매칭 완료 | Ratio Test 통과: {len(good_matches)}개')

# 5. RANSAC + Homography 계산 및 최종 시각화
if len(good_matches) > 4:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # RANSAC: 4점 무작위 샘플 → homography 추정 → inlier 검증 반복
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 3.0)
    matches_mask = mask.ravel().tolist()

    # Inliers만 추출
    inlier_matches = [good_matches[i] for i in range(len(matches_mask)) if matches_mask[i] == 1]
    print(f'RANSAC 검증 완료 | 최종 인라이어(Inliers): {len(inlier_matches)}개')

    res_img = cv2.drawMatches(img1_gray, kp1, img2_gray, kp2,
                              inlier_matches[:500], None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    print('\n--- [실습 7 결과: RANSAC 매칭 결과] ---')
    cv2_imshow(res_img)
else:
    print('매칭점이 부족하여 호모그래피를 계산할 수 없습니다.')

## [실습 8] Stereo Block Matching (StereoBM, SAD 기반)

**목표:** OpenCV의 `cv2.StereoBM_create`(SAD: Sum of Absolute Differences 기반)를 사용해 좌/우 스테레오 영상에서 양안시차(disparity) 지도를 계산하고 시각화합니다.

In [ ]:
# 1. Aloe 스테레오 페어 다운로드
url1 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeL.jpg'
url2 = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/aloeR.jpg'
urllib.request.urlretrieve(url1, 'aloeL.jpg')
urllib.request.urlretrieve(url2, 'aloeR.jpg')

imgL_color = cv2.imread('aloeL.jpg')
imgL_gray = cv2.imread('aloeL.jpg', cv2.IMREAD_GRAYSCALE)
imgR_gray = cv2.imread('aloeR.jpg', cv2.IMREAD_GRAYSCALE)

# 원본 해상도 저장
orig_h, orig_w = imgL_gray.shape

# 2. StereoBM 설정 (SAD 기반)
# numDisparities: 검색할 시차 범위 (16의 배수여야 함)
# blockSize: 매칭 블록 크기 (홀수, 보통 5~21)
num_disp = 128
block_size = 15
stereo = cv2.StereoBM_create(numDisparities=num_disp, blockSize=block_size)

# 3. Disparity 계산 + 소요시간 측정
start_time = time.time()
disparity = stereo.compute(imgL_gray, imgR_gray)
elapsed_time = time.time() - start_time

# StereoBM 결과는 16배 스케일 정수이므로 실제 픽셀 단위로 변환
disparity_float = disparity.astype(np.float32) / 16.0

# 4. 시각화를 위한 0~255 정규화
disparity_vis = cv2.normalize(disparity, None, 0, 255,
                              cv2.NORM_MINMAX, cv2.CV_8U)

# 5. 결과 출력
print('--- [실습 8 결과: StereoBM Disparity Map] ---')
print(f'이미지 크기: {orig_w}x{orig_h}')
print(f'소요 시간: {elapsed_time:.4f}초')

plt.figure(figsize=(20, 10))

# 5-1. 왼쪽 원본 이미지
plt.subplot(1, 2, 1)
plt.title('Original Left Image')
plt.imshow(cv2.cvtColor(imgL_color, cv2.COLOR_BGR2RGB))
plt.axis('off')

# 5-2. 원본 크기의 Disparity Map (jet colormap)
plt.subplot(1, 2, 2)
plt.title(f'Full-Res SAD Disparity Map\n(NumDisp: {num_disp}, Block: {block_size})')
plt.imshow(disparity_vis, cmap='jet')
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis('off')

plt.tight_layout()
plt.show()

## [실습 9] 2.5D 영상 (Disparity → Point Cloud → PLY 저장)

**목표:** 실습 8에서 생성한 disparity map을 이용해 카메라 좌표계의 3D 점(Point Cloud)을 역투영하고, PLY(Stanford Polygon) 포맷으로 저장하여 Meshlab 등에서 확인할 수 있게 합니다.

- 깊이 변환 공식: $Z = (f \cdot B) / d$
- $X = (u - W/2) \cdot Z / f$, $Y = (v - H/2) \cdot Z / f$

In [ ]:
# 1. 실습 8에서 생성한 데이터(imgL_color, disparity_float) 활용
scale = 1.0  # 처리량 조절용 (1.0 = 원본 크기)
img_small = cv2.resize(imgL_color, None, fx=scale, fy=scale)
disp_small = cv2.resize(disparity_float, None, fx=scale, fy=scale)

h, w = disp_small.shape

# 2. 가상의 카메라 내부 파라미터 정의
f = 0.8 * w   # 가상의 초점 거리(focal length)
B = 1.0       # 가상의 두 카메라 간격(baseline)

# 3. 픽셀 좌표 그리드 생성 및 유효 disparity 마스크 정의
x_coords, y_coords = np.meshgrid(np.arange(w), np.arange(h))
mask = disp_small > 0.5  # 너무 작은 값은 노이즈로 간주하여 제외

# 4. 3D 좌표 계산 (Z = f * B / d, X/Y는 핀홀 카메라 모델 역투영)
z = np.zeros_like(disp_small)
z[mask] = (f * B) / disp_small[mask]

x = (x_coords - w/2) * z / f
y = (y_coords - h/2) * z / f

# 5. 시각화 데이터 준비 (유효 픽셀만 1차원으로 펼침)
points_x = x[mask].ravel()
points_y = y[mask].ravel()
points_z = z[mask].ravel()

# 6. 색상 정보 추출 (BGR → RGB, 0~1 정규화)
colors = cv2.cvtColor(img_small, cv2.COLOR_BGR2RGB)[mask].reshape(-1, 3) / 255.0

print(f'유효 포인트 개수: {len(points_x)}')

In [ ]:
def save_point_cloud_ply(filename, points_x, points_y, points_z, colors):
    """
    포인트 클라우드 데이터를 PLY(ASCII) 파일로 저장
    points_x/y/z: 1차원 numpy 배열 (좌표)
    colors: 1차원 numpy 배열 (0~1 범위의 RGB)
    """
    # 색상을 0~255 범위의 정수로 변환
    colors_int = (colors * 255).astype(np.uint8)

    # PLY 헤더 작성
    num_points = len(points_x)
    header = f"""ply
format ascii 1.0
element vertex {num_points}
property float x
property float y
property float z
property uchar red
property uchar green
property uchar blue
end_header
"""
    # 헤더 + 정점 데이터 기록
    with open(filename, 'w') as f:
        f.write(header)
        for i in range(num_points):
            f.write(f'{points_x[i]} {points_y[i]} {points_z[i]} '
                    f'{colors_int[i, 0]} {colors_int[i, 1]} {colors_int[i, 2]}\n')

    print(f"성공: '{filename}' 파일이 저장되었습니다. (총 {num_points}개의 점)")

# 7. PLY 파일로 저장 (Meshlab 등에서 열어 3D 시각화 가능)
save_point_cloud_ply('aloe_2_5d.ply', points_x, points_y, points_z, colors)

In [ ]:
# 8. 노트북 내에서 3D 산점도로 미리보기 (다운샘플링)
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (3D projection 활성화)

step = max(1, len(points_x) // 20000)  # 너무 많으면 20K 포인트 정도로 감속
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(points_x[::step], points_y[::step], points_z[::step],
           c=colors[::step], s=0.5)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z (Depth)')
ax.set_title('2.5D Point Cloud Preview (Aloe)')
ax.view_init(elev=-70, azim=-90)  # 카메라 시점에 가깝게 회전
print('--- [실습 9 결과: 2.5D Point Cloud Preview] ---')
plt.show()